# Flickr30k T1→S1 Hybrid KD Protocol

Khóa trước phương pháp kết hợp **Sequence KD** và
**Dense Mapped Logit-KD**:

\[
\mathcal{L}_{hybrid}
=0{,}50\mathcal{L}_{CE}
+0{,}25\mathcal{L}_{seq}
+0{,}25T^2\mathcal{L}_{KL}^{mapped},\qquad T=4.
\]

Đây là thí nghiệm hậu nghiệm chỉ dùng Train/Validation. Held-out Test
Flickr30k đã được mở cho nhánh T3→S3 đã khóa, vì vậy không được dùng Test
đó để chọn trọng số, checkpoint hoặc đưa ra một tuyên bố xác nhận mới.

## Kaggle input bắt buộc — đúng 3 input

1. Output `F30K_Final_Audit`.
2. Output `F30K_T1S1_SeqKD_Aggregate`.
3. Output `F30K_T1S1_DenseMap_Aggregate`.

Runtime: CPU/Accelerator None, Internet Off, khoảng 1–2 phút.

## Cell 1 — Lock phương pháp và ranh giới khoa học

In [1]:
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

EXPERIMENT_ID = "F30K_T1S1_Hybrid_Protocol"
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KLCN_OUTPUT_ROOT", "/kaggle/working")) / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

def write_json(path, payload):
    temporary = Path(path).with_suffix(Path(path).suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

source_names = {
    "final_audit": "F30K_Final_Audit_RESULT.json",
    "sequence_aggregate": "F30K_T1S1_SeqKD_Aggregate_RESULT.json",
    "dense_aggregate": "F30K_T1S1_DenseMap_Aggregate_RESULT.json",
}
sources = {}
for key, name in source_names.items():
    path = unique_named(name)
    payload = read_json(path)
    assert payload["status"] in {"complete", "passed"}
    sources[key] = {
        "path": str(path),
        "sha256": sha256(path),
        "payload": payload,
    }

sequence = sources["sequence_aggregate"]["payload"]
dense_map = sources["dense_aggregate"]["payload"]
assert sequence["seeds"] == dense_map["seeds"] == [42, 123, 2026]
assert sequence["pair_id"] == "T1S1_SeqKD"
assert dense_map["pair_id"] == "T1S1_DenseMap"
assert sequence["test_set_used"] is dense_map["test_set_used"] is False

protocol = {
    "schema_version": "f30k_t1s1_hybrid_protocol_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "locked_validation_only",
    "experiment_id": EXPERIMENT_ID,
    "teacher": "T1 ExpansionNet v2",
    "student": "S1 MobileNetV3-Large",
    "method": "Hybrid Sequence KD + Dense Mapped Logit-KD",
    "loss": {
        "formula": "0.50*CE_human + 0.25*CE_sequence + 0.25*T^2*KL_mapped",
        "alpha_human": 0.50,
        "alpha_sequence": 0.25,
        "alpha_dense_mapped": 0.25,
        "temperature": 4.0,
        "weights_locked_before_training": True,
        "weight_search_authorized": False,
    },
    "seeds": [42, 123, 2026],
    "maximum_epochs": 20,
    "early_stopping": {
        "enabled": True,
        "selection_metric": "Validation CIDEr-D",
        "minimum_completed_epochs": 5,
        "patience": 4,
        "minimum_cider_improvement": 0.001,
    },
    "batch": {"physical": 1, "gradient_accumulation": 64, "effective": 64},
    "development_split": "Flickr30k Train",
    "selection_split": "Flickr30k Validation (1014 images)",
    "study_status": "post_hoc_validation_only",
    "reason": (
        "Flickr30k held-out Test was already opened for the locked T3S3 branch. "
        "This follow-up cannot reuse that Test for tuning, selection, promotion, "
        "or a new final unbiased claim."
    ),
    "test_set_status": "previously_opened_for_locked_T3S3_only",
    "test_set_used_by_followup": False,
    "test_evaluation_authorized": False,
    "promotion_to_existing_final_lock_authorized": False,
    "independent_unseen_evaluation_required_for_confirmatory_claim": True,
    "component_validation_references": {
        "sequence_kd_delta_cider_d_mean": sequence["summary"]["CIDEr_D"]["delta_mean"],
        "dense_mapped_delta_cider_d_mean": dense_map["summary"]["CIDEr_D"]["delta_mean"],
    },
    "source_sha256": {
        key: value["sha256"] for key, value in sources.items()
    },
    "next_phase_authorized": [
        "F30K_T1S1_Hybrid_42",
        "F30K_T1S1_Hybrid_123",
        "F30K_T1S1_Hybrid_2026",
    ],
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json(result_path, protocol)
manifest = {
    "schema_version": "f30k_t1s1_hybrid_protocol_manifest_v1",
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "artifacts": [{
        "relative_path": result_path.name,
        "size_bytes": result_path.stat().st_size,
        "sha256": sha256(result_path),
    }],
    "test_set_status": protocol["test_set_status"],
    "test_set_used_by_followup": False,
}
write_json(WORK_ROOT / "MANIFEST.json", manifest)

print("Protocol status          : LOCKED / VALIDATION ONLY")
print("Loss                     :", protocol["loss"]["formula"])
print("Seeds                    :", protocol["seeds"])
print("Parallel seed runs       : AUTHORIZED")
print("Old held-out Test        : NOT AUTHORIZED FOR THIS FOLLOW-UP")
print("Next                     :", protocol["next_phase_authorized"])

Protocol status          : LOCKED / VALIDATION ONLY
Loss                     : 0.50*CE_human + 0.25*CE_sequence + 0.25*T^2*KL_mapped
Seeds                    : [42, 123, 2026]
Parallel seed runs       : AUTHORIZED
Old held-out Test        : NOT AUTHORIZED FOR THIS FOLLOW-UP
Next                     : ['F30K_T1S1_Hybrid_42', 'F30K_T1S1_Hybrid_123', 'F30K_T1S1_Hybrid_2026']
